In [1]:
from pydub import AudioSegment
import numpy as np


/usr/local/lib/python3.11/site-packages/pydub/utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


In [4]:

def generate_spooky_sound(duration_sec=10, sample_rate=44100):
    """
    Genera un sonido tétrico procedural de 'duration_sec' segundos.
    """
    t = np.linspace(0, duration_sec, int(sample_rate * duration_sec), False)

    # Tonos graves oscuros
    freq1 = 40  # Hz
    freq2 = 60  # Hz

    wave1 = np.sin(2 * np.pi * freq1 * t) * 0.5
    wave2 = np.sin(2 * np.pi * freq2 * t) * 0.5

    # Ruidos tipo viento
    noise = np.random.normal(0, 0.1, wave1.shape)

    # Mezcla
    audio_wave = (wave1 + wave2 + noise) * 32767
    audio_wave = np.clip(audio_wave, -32768, 32767).astype(np.int16)

    # Convertir a AudioSegment
    audio_segment = AudioSegment(
        audio_wave.tobytes(), 
        frame_rate=sample_rate,
        sample_width=2,  # 2 bytes por muestra
        channels=1
    )

    # --- Agregar eco correctamente ---
    delay_ms = 250  # 250 milisegundos de eco
    delayed = AudioSegment.silent(duration=delay_ms) + audio_segment
    reverb = audio_segment.overlay(delayed)

    return reverb


In [5]:
# Guardarlo
spooky_music = generate_spooky_sound(duration_sec=10)
spooky_music.export("spooky_music.wav", format="wav")

<_io.BufferedRandom name='spooky_music.wav'>

In [25]:

def generate_spooky_sound(duration_sec=10, sample_rate=44100, add_heartbeat=True):
    """
    Genera un sonido tétrico procedural de 'duration_sec' segundos.
    Si add_heartbeat es True, agrega latidos graves.
    """
    t = np.linspace(0, duration_sec, int(sample_rate * duration_sec), False)

    # Tonos graves oscuros
    freq1 = 40  # Hz
    freq2 = 60  # Hz

    # wave1 = np.sin(2 * np.pi * freq1 * t) * 0.5
    # wave2 = np.sin(2 * np.pi * freq2 * t) * 0.5

    wave1 = np.sin(2 * np.pi * freq1 * t) * np.sin(0.25 * np.pi * t) #* 0.5  # modulación
    wave2 = np.sin(2 * np.pi * freq2 * t) * np.cos(0.15 * np.pi * t) #* 0.5

    # Ruidos tipo viento
    noise = np.random.normal(0, 0.1, wave1.shape)

    # Mezcla
    # audio_wave = (noise) * 32767
    audio_wave = (wave1) * 32767
    # audio_wave = (wave2) * 32767
    # audio_wave = (wave1 + wave2) * 32767
    # audio_wave = (wave1 + wave2 + noise) * 32767
    audio_wave = np.clip(audio_wave, -32768, 32767).astype(np.int16)

    # Convertir a AudioSegment
    audio_segment = AudioSegment(
        audio_wave.tobytes(), 
        frame_rate=sample_rate,
        sample_width=2,
        channels=1
    )

    # --- Agregar eco básico ---
    delay_ms = 250
    delayed = AudioSegment.silent(duration=delay_ms) + audio_segment
    reverb = audio_segment.overlay(delayed)

    # --- Agregar latido oscuro ---
    if add_heartbeat:
        reverb = add_dark_heartbeat(reverb, beat_interval_sec=0.8)

    return reverb

def add_dark_heartbeat(audio, beat_interval_sec=0.8):
    """
    Agrega un latido grave repetido cada beat_interval_sec segundos.
    """
    heartbeat = generate_heartbeat(duration_ms=150)
    output = audio

    beat_interval_ms = int(beat_interval_sec * 1000)
    total_duration_ms = len(audio)

    for pos in range(0, total_duration_ms, beat_interval_ms):
        output = output.overlay(heartbeat, position=pos)

    return output

# def generate_heartbeat(duration_ms=150, freq=50):
#     """
#     Genera un pulso grave simple para simular un latido.
#     """
#     sample_rate = 44100
#     t = np.linspace(0, duration_ms / 1000, int(sample_rate * (duration_ms / 1000)), False)
#     wave = (np.sin(2 * np.pi * freq * t) * np.exp(-5 * t))  # pulso decae rápidamente
#     audio_wave = (wave * 32767).astype(np.int16)

#     heartbeat = AudioSegment(
#         audio_wave.tobytes(),
#         frame_rate=sample_rate,
#         sample_width=2,
#         channels=1
#     )

#     # Ajustar volumen del latido (más bajo que el fondo)
#     heartbeat = heartbeat - 6  # bajar 6 dB

#     return heartbeat

def generate_heartbeat(duration_ms=250, freq=60):
    sample_rate = 44100
    t = np.linspace(0, duration_ms / 1000, int(sample_rate * duration_ms / 1000), False)

    # Pulso modulado (envolvente exponencial)
    envelope = np.exp(-8 * t)  # decay más lento
    wave = np.sin(2 * np.pi * freq * t) * envelope

    audio_wave = (wave * 32767).astype(np.int16)

    heartbeat = AudioSegment(
        audio_wave.tobytes(),
        frame_rate=sample_rate,
        sample_width=2,
        channels=1
    )

    return heartbeat - 3  # volumen moderado


In [26]:
from datetime import datetime
now = datetime.now()
formatted_time = now.strftime("%Y%m%d%H%M%S")

spooky_music = generate_spooky_sound(duration_sec=30)
spooky_music.export("latido_music"+formatted_time+".wav", format="wav")

<_io.BufferedRandom name='latido_music20250429044303.wav'>

# Latido independiente

In [ ]:

def generate_spooky_background(duration_sec=10, sample_rate=44100):
    """
    Genera solo el fondo tétrico continuo, sin latido.
    """
    t = np.linspace(0, duration_sec, int(sample_rate * duration_sec), False)

    freq1 = 40  # Hz
    freq2 = 60  # Hz

    # Añadimos modulación lenta para evitar "trrrrr"
    wave1 = np.sin(2 * np.pi * freq1 * t) * np.sin(0.25 * np.pi * t) * 0.5
    wave2 = np.sin(2 * np.pi * freq2 * t) * np.cos(0.15 * np.pi * t) * 0.5

    noise = np.random.normal(0, 0.05, wave1.shape)

    audio_wave = (wave1 + wave2 + noise) * 32767
    audio_wave = np.clip(audio_wave, -32768, 32767).astype(np.int16)

    background = AudioSegment(
        audio_wave.tobytes(), 
        frame_rate=sample_rate,
        sample_width=2,
        channels=1
    )

    return background

def generate_heartbeat_track(duration_sec=10, beat_interval_sec=1.2, volume_boost_db=6):
    """
    Genera solo la pista de latidos separados.
    """
    sample_rate = 44100
    silence = AudioSegment.silent(duration=int(duration_sec * 1000))

    heartbeat = generate_heartbeat(duration_ms=250, freq=60, volume_boost_db=volume_boost_db)

    # Poner un latido cada beat_interval_sec
    for pos_ms in range(0, len(silence), int(beat_interval_sec * 1000)):
        silence = silence.overlay(heartbeat, position=pos_ms)

    return silence

def generate_heartbeat(duration_ms=250, freq=60, volume_boost_db=6):
    """
    Genera un solo latido de duración corta.
    """
    sample_rate = 44100
    t = np.linspace(0, duration_ms / 1000, int(sample_rate * (duration_ms / 1000)), False)

    envelope = np.exp(-8 * t)
    wave = np.sin(2 * np.pi * freq * t) * envelope

    audio_wave = (wave * 32767).astype(np.int16)

    heartbeat = AudioSegment(
        audio_wave.tobytes(),
        frame_rate=sample_rate,
        sample_width=2,
        channels=1
    )

    return heartbeat + volume_boost_db  # aumentar volumen del latido

def mix_tracks(background, heartbeat_track):
    """
    Mezcla el fondo y los latidos separados.
    """
    return background.overlay(heartbeat_track)

In [30]:
from datetime import datetime
now = datetime.now()
formatted_time = now.strftime("%Y%m%d%H%M%S")

# --- USO ---
duration_sec = 30

# Generar las dos pistas independientes
background = generate_spooky_background(duration_sec)
heartbeat_track = generate_heartbeat_track(duration_sec)

# Opcional: Puedes exportarlas separadas si quieres
# background.export("background_only.wav", format="wav")
# heartbeat_track.export("heartbeat_only.wav", format="wav")

# Mezclar para crear la versión final
final_mix = mix_tracks(background, heartbeat_track)
final_mix.export("final_spooky_mix"+formatted_time+".wav", format="wav")

<_io.BufferedRandom name='final_spooky_mix20250429044657.wav'>

# Melodia de piano

In [44]:
def generate_piano_note(freq=440, duration_ms=400, sample_rate=44100, volume_db=-6):
    """
    Genera una nota tipo piano simple con caída rápida (envelope).
    """
    t = np.linspace(0, duration_ms / 1000, int(sample_rate * duration_ms / 1000), False)
    envelope = np.exp(-3 * t)  # Decay típico de piano
    wave = np.sin(2 * np.pi * freq * t) * envelope

    audio_wave = (wave * 32767).astype(np.int16)

    note = AudioSegment(
        audio_wave.tobytes(),
        frame_rate=sample_rate,
        sample_width=2,
        channels=1
    )

    return note + volume_db


def generate_piano_melody():
    """
    Crea una melodía tétrica y lenta con notas menores.
    """
    notes_freq = {
        'C4': 261.63,
        'Eb4': 311.13,
        'F4': 349.23,
        'G4': 392.00,
        'Bb3': 233.08,
        'C5': 523.25
    }

    # Una melodía lenta y sombría en C menor
    melody_sequence = [
        ('C4', 600), ('Eb4', 400), ('F4', 600),
        ('C4', 600), ('Bb3', 800), ('G4', 500),
        ('F4', 400), ('C5', 1000)
    ]

    full_melody = AudioSegment.silent(duration=0)
    for note_name, duration in melody_sequence:
        freq = notes_freq[note_name]
        note = generate_piano_note(freq=freq, duration_ms=duration)
        full_melody += note + AudioSegment.silent(duration=100)  # pequeño silencio entre notas

    return full_melody

In [46]:
from datetime import datetime
now = datetime.now()
formatted_time = now.strftime("%Y%m%d%H%M%S")

# Supón que ya tienes:
# - background (solo fondo tétrico)
# - heartbeat_track (latidos)

piano_melody = generate_piano_melody()
piano_melody = piano_melody[:len(background)]  # Recorta si es más largo

piano_melody.export("piano_only.wav", format="wav")

# Mezclar todo
# combined = background.overlay(heartbeat_track).overlay(piano_melody - 3)  # bajamos piano un poco

# combined.export("final_spooky_with_piano"+formatted_time+".wav", format="wav")

<_io.BufferedRandom name='piano_only.wav'>

# Usando Theremin

In [34]:
def generate_theremin_note(base_freq=440, duration_ms=800, vibrato_freq=5, vibrato_amount=5, sample_rate=44100, volume_db=-6):
    """
    Genera un sonido tipo theremín con vibrato.
    """
    t = np.linspace(0, duration_ms / 1000, int(sample_rate * duration_ms / 1000), False)

    # Vibrato: frecuencia base modulada por una onda pequeña
    vibrato = np.sin(2 * np.pi * vibrato_freq * t) * vibrato_amount
    freq = base_freq + vibrato

    wave = np.sin(2 * np.pi * freq * t)

    # Envolvente tipo fade in y fade out
    envelope = np.clip(np.sin(np.pi * t / (duration_ms / 1000)), 0, 1)
    wave *= envelope

    audio_wave = (wave * 32767).astype(np.int16)

    note = AudioSegment(
        audio_wave.tobytes(),
        frame_rate=sample_rate,
        sample_width=2,
        channels=1
    )

    return note + volume_db

def generate_theremin_melody():
    """
    Crea una melodía tétrica tipo theremín.
    """
    notes_freq = {
        'C4': 261.63,
        'D4': 293.66,
        'Eb4': 311.13,
        'G3': 196.00,
        'Bb3': 233.08,
        'F4': 349.23,
    }

    # Una secuencia lenta y oscura
    melody_sequence = [
        ('G3', 1000), ('Bb3', 800), ('C4', 1200),
        ('D4', 1000), ('Eb4', 800), ('F4', 1200)
    ]

    full_melody = AudioSegment.silent(duration=0)
    for note_name, duration in melody_sequence:
        freq = notes_freq[note_name]
        note = generate_theremin_note(base_freq=freq, duration_ms=duration)
        full_melody += note + AudioSegment.silent(duration=150)  # pequeños silencios

    return full_melody

In [ ]:
from datetime import datetime
now = datetime.now()
formatted_time = now.strftime("%Y%m%d%H%M%S")


# Asumiendo que ya tienes:
# - background
# - heartbeat_track

theremin_melody = generate_theremin_melody()
theremin_melody = theremin_melody[:len(background)]  # Recortarlo si es más largo


# Mezclar todo
combined = background.overlay(heartbeat_track).overlay(theremin_melody - 6)  # bajamos un poco el theremín

combined.export("final_spooky_with_theremin"+formatted_time+".wav", format="wav")

<_io.BufferedRandom name='final_spooky_with_theremin20250429045237.wav'>

In [36]:
theremin_melody.export("theremin_only.wav", format="wav")

<_io.BufferedRandom name='theremin_only.wav'>

# Melodia mas elaborada con el theremin

In [37]:
def generate_theremin_note(base_freq=440, duration_ms=800, vibrato_freq=5, vibrato_amount=6, sample_rate=44100, volume_db=-6):
    """
    Genera un sonido tipo theremín con vibrato.
    """
    t = np.linspace(0, duration_ms / 1000, int(sample_rate * duration_ms / 1000), False)

    # Vibrato: frecuencia base modulada
    vibrato = np.sin(2 * np.pi * vibrato_freq * t) * vibrato_amount
    freq = base_freq + vibrato

    wave = np.sin(2 * np.pi * freq * t)

    # Envolvente: fade in/out natural
    envelope = np.clip(np.sin(np.pi * t / (duration_ms / 1000)), 0, 1)
    wave *= envelope

    audio_wave = (wave * 32767).astype(np.int16)

    note = AudioSegment(
        audio_wave.tobytes(),
        frame_rate=sample_rate,
        sample_width=2,
        channels=1
    )

    return note + volume_db

In [38]:
def generate_elaborate_theremin_melody():
    """
    Genera una melodía de theremín más compleja, estilo película de terror.
    """
    notes_freq = {
        'G3': 196.00,
        'A3': 220.00,
        'Bb3': 233.08,
        'C4': 261.63,
        'D4': 293.66,
        'Eb4': 311.13,
        'F4': 349.23,
        'G4': 392.00,
        'A4': 440.00,
    }

    # Frase 1 (sube misteriosamente)
    phrase1 = [
        ('G3', 1000), ('A3', 800), ('Bb3', 600), ('C4', 1200)
    ]

    # Frase 2 (más tensa, notas cortas y más agudas)
    phrase2 = [
        ('D4', 400), ('Eb4', 400), ('F4', 600), ('D4', 500)
    ]

    # Frase 3 (un clímax, notas más largas y más altas)
    phrase3 = [
        ('F4', 1000), ('G4', 1200), ('A4', 1500)
    ]

    all_phrases = phrase1 + [('silence', 400)] + phrase2 + [('silence', 600)] + phrase3

    full_melody = AudioSegment.silent(duration=0)

    for note_name, duration in all_phrases:
        if note_name == 'silence':
            full_melody += AudioSegment.silent(duration=duration)
        else:
            freq = notes_freq[note_name]
            note = generate_theremin_note(base_freq=freq, duration_ms=duration)
            full_melody += note + AudioSegment.silent(duration=80)  # pequeños respiros

    return full_melody

In [39]:
# Genera todo
theremin_melody = generate_elaborate_theremin_melody()
theremin_melody = theremin_melody[:len(background)]  # ajustar duración si es más largo

theremin_melody.export("theremin_only_2.wav", format="wav")

# Mezclar pistas
# combined = background.overlay(heartbeat_track).overlay(theremin_melody - 4)

# combined.export("final_spooky_with_elaborate_theremin.wav", format="wav")

<_io.BufferedRandom name='theremin_only_2.wav'>

# Mejorando theremin

In [40]:
def generate_theremin_glissando(start_freq=130, end_freq=220, duration_ms=2000, vibrato_freq=4, vibrato_amount=5, sample_rate=44100, volume_db=-6):
    """
    Genera una nota tipo theremín que desliza su frecuencia de inicio a fin (glissando).
    """
    t = np.linspace(0, duration_ms / 1000, int(sample_rate * duration_ms / 1000), False)

    # Línea de frecuencias de start a end
    freq = np.linspace(start_freq, end_freq, t.size)

    # Aplicar vibrato sobre el cambio de frecuencia
    vibrato = np.sin(2 * np.pi * vibrato_freq * t) * vibrato_amount
    freq += vibrato

    wave = np.sin(2 * np.pi * freq * t)

    # Envolvente (desvanecido inicial y final suave)
    envelope = np.clip(np.sin(np.pi * t / (duration_ms / 1000)), 0, 1)
    wave *= envelope

    audio_wave = (wave * 32767).astype(np.int16)

    note = AudioSegment(
        audio_wave.tobytes(),
        frame_rate=sample_rate,
        sample_width=2,
        channels=1
    )

    return note + volume_db

def generate_continuous_theremin_melody():
    """
    Genera una melodía de theremín lenta, grave y continua con glissandos.
    """
    # Lista de (frecuencia inicial, frecuencia final, duración)
    melody = [
        (130, 150, 3000),  # G2 a A2
        (150, 170, 2500),  # A2 a B2
        (170, 160, 2000),  # B2 baja a Bb2
        (160, 196, 3500),  # Bb2 sube a G3
        (196, 175, 2500),  # G3 baja un poco
        (175, 220, 4000)   # Sube más a A3
    ]

    full_melody = AudioSegment.silent(duration=0)

    for start_freq, end_freq, duration in melody:
        note = generate_theremin_glissando(
            start_freq=start_freq,
            end_freq=end_freq,
            duration_ms=duration
        )
        full_melody += note  # Sin silencios para que sea continuo

    return full_melody



In [42]:

theremin_melody = generate_continuous_theremin_melody()
theremin_melody = theremin_melody[:len(background)]  # ajustamos duración


theremin_melody.export("theremin_only_3.wav", format="wav")


# Mezclar todo
combined = background.overlay(heartbeat_track).overlay(theremin_melody - 5)

combined.export("final_spooky_with_continuous_theremin.wav", format="wav")

<_io.BufferedRandom name='final_spooky_with_continuous_theremin.wav'>

# Solo piano

In [47]:
from pydub import AudioSegment
from pydub.generators import Sine

In [ ]:
def create_note(frequency, duration):
    return Sine(frequency).to_audio_segment(duration=duration)

# Notas musicales en Hz (piano de 4 octavas)
notes = {
    'C4': 261.63,
    'C#4': 277.18,
    'D4': 293.66,
    'D#4': 311.13,
    'E4': 329.63,
    'F4': 349.23,
    'F#4': 369.99,
    'G4': 392.00,
    'G#4': 415.30,
    'A4': 440.00,
    'A#4': 466.16,
    'B4': 493.88,
    'C5': 523.25,
}

# Duración de cada nota (en milisegundos)
note_duration = 400  # 400 ms para cada nota

# Melodía: una lista de notas
melody = [
    'C4', 'D4', 'E4', 'C4', 'E4', 'D4', 'C4', 'C4', 
    'E4', 'F4', 'G4', 'C5', 'A4', 'G4', 'F4', 'E4', 'D4',
    'C4', 'D4', 'E4', 'C4', 'E4', 'F4', 'G4', 'C5'
]

# Crear la melodía
melody_audio = AudioSegment.silent(duration=0)  # Empezamos con silencio

# Añadir las notas una a una
for note in melody:
    note_audio = create_note(notes[note], note_duration)
    melody_audio += note_audio

# Exportar la melodía a un archivo WAV
melody_audio.export("melody_piano.wav", format="wav")

print("Melodía generada y exportada como 'melody_piano.wav'")

Melodía generada y exportada como 'melody_piano.wav'


In [49]:
# Función para crear una nota con frecuencia específica (en Hz) y duración (en milisegundos)
def create_note(frequency, duration):
    return Sine(frequency).to_audio_segment(duration=duration)

# Notas musicales en Hz (piano de 4 octavas, usando notas graves)
notes = {
    'C3': 130.81,
    'C#3': 138.59,
    'D3': 146.83,
    'D#3': 155.56,
    'E3': 164.81,
    'F3': 174.61,
    'F#3': 185.00,
    'G3': 196.00,
    'G#3': 207.65,
    'A3': 220.00,
    'A#3': 233.08,
    'B3': 246.94,
    'C4': 261.63,
    'D4': 293.66,
}

# Duración de cada nota (en milisegundos), usando duraciones largas
note_duration = 800  # 800 ms para cada nota, más largas

# Melodía más tétrica: secuencia de notas más graves y oscuros
melody = [
    'C3', 'D3', 'E3', 'D3', 'C3', 'C3', 'C3', 'D3', 
    'D#3', 'E3', 'F3', 'C3', 'D3', 'E3', 'D3', 'C3', 
    'C3', 'A3', 'A3', 'G#3', 'G3', 'F3', 'D3', 'C3',
    'G3', 'F#3', 'E3', 'C3'
]

# Crear la melodía
melody_audio = AudioSegment.silent(duration=0)  # Empezamos con silencio

# Añadir las notas una a una
for note in melody:
    note_audio = create_note(notes[note], note_duration)
    melody_audio += note_audio

# Exportar la melodía a un archivo WAV
melody_audio.export("melody_tettrica_piano.wav", format="wav")

print("Melodía tétrica generada y exportada como 'melody_tettrica_piano.wav'")

Melodía tétrica generada y exportada como 'melody_tettrica_piano.wav'


In [51]:

# Función para crear una nota con frecuencia específica (en Hz) y duración (en milisegundos)
def create_note(frequency, duration):
    return Sine(frequency).to_audio_segment(duration=duration)

# Notas musicales en Hz (piano de 4 octavas, usando notas graves)
notes = {
    'C3': 130.81,
    'C#3': 138.59,
    'D3': 146.83,
    'D#3': 155.56,
    'E3': 164.81,
    'F3': 174.61,
    'F#3': 185.00,
    'G3': 196.00,
    'G#3': 207.65,
    'A3': 220.00,
    'A#3': 233.08,
    'B3': 246.94,
    'C4': 261.63,
    'D4': 293.66,
}

# Duración de cada nota (en milisegundos), usando duraciones largas
note_duration = 800  # 800 ms para cada nota, más largas

# Melodía más tétrica: secuencia de notas más graves y oscuros
melody = [
    'C3', 'D3', 'E3', 'D3', 'C3', 'C3', 'C3', 'D3', 
    'D#3', 'E3', 'F3', 'C3', 'D3', 'E3', 'D3', 'C3', 
    'C3', 'A3', 'A3', 'G#3', 'G3', 'F3', 'D3', 'C3',
    'G3', 'F#3', 'E3', 'C3'
]

# Función para crear la melodía completa
def create_melody():
    melody_audio = AudioSegment.silent(duration=0)  # Empezamos con silencio
    for note in melody:
        note_audio = create_note(notes[note], note_duration).apply_gain(-6)
        melody_audio += note_audio
    return melody_audio

# Parámetro de duración total en segundos
total_duration_seconds = 60  # Duración total deseada en segundos

# Crear la melodía completa
melody_audio = create_melody()

# Duración de una repetición (10 segundos)
repeat_duration = 10 * 1000  # 10 segundos en milisegundos

# Calcular cuántas veces se repite la melodía en el tiempo total
num_repeats = total_duration_seconds // 10

# Repetir la melodía hasta alcanzar la duración total
final_audio = AudioSegment.silent(duration=2)  # Empezamos con silencio
for _ in range(num_repeats):
    final_audio += melody_audio


from datetime import datetime
now = datetime.now()
formatted_time = now.strftime("%Y%m%d%H%M%S")

# Exportar la melodía repetida a un archivo WAV
final_audio.export("melody_repeated_piano"+formatted_time+".wav", format="wav")

print(f"Melodía generada con duración total de {total_duration_seconds} segundos y exportada como 'melody_repeated_piano.wav'")

Melodía generada con duración total de 60 segundos y exportada como 'melody_repeated_piano.wav'


# Aún más tétrica

In [55]:

# Función para crear una nota con frecuencia específica (en Hz) y duración (en milisegundos)
def create_note(frequency, duration):
    return Sine(frequency).to_audio_segment(duration=duration)

# Notas musicales más graves (segunda octava)
notes = {
    'C2': 65.41,
    'C#2': 69.30,
    'D2': 73.42,
    'D#2': 77.78,
    'E2': 82.41,
    'F2': 87.31,
    'F#2': 92.50,
    'G2': 98.00,
    'G#2': 103.83,
    'A2': 110.00,
    'A#2': 116.54,
    'B2': 123.47,
    'C3': 130.81,
    'D3': 146.83,
}

# Duración de cada nota (más lenta, 1200 ms)
# note_duration = 1200
note_duration = 1000

# Melodía tétrica con tonos muy graves
melody = [
    'C2', 'D2', 'D#2', 'D2', 'C2', 'C2', 'C2', 'D2',
    'D#2', 'E2', 'F2', 'C2', 'D2', 'E2', 'D2', 'C2',
    'A2', 'A2', 'G#2', 'G2', 'F2', 'D2', 'C2'
]

# Función para crear la melodía completa con pausas
# def create_melody():
#     melody_audio = AudioSegment.silent(duration=0)
#     short_pause = AudioSegment.silent(duration=300)  # Pausa de 300 ms entre notas
#     for note in melody:
#         note_audio = create_note(notes[note], note_duration).apply_gain(-6)  # Baja volumen individualmente
#         melody_audio += note_audio #+ short_pause  # Añade pausa después de cada nota
#     return melody_audio

def create_melody():
    melody_audio = AudioSegment.silent(duration=0)
    short_pause = AudioSegment.silent(duration=300)  # Pausa de 300 ms entre notas
    for note in melody:
        # Creamos la nota con un pequeño fade_out para evitar chasquido
        note_audio = create_note(notes[note], note_duration).fade_out(100).apply_gain(-6)
        melody_audio += note_audio #+ short_pause
    return melody_audio


# Parámetro de duración total en segundos
total_duration_seconds = 20  # Duración total deseada

# Crear la melodía base
melody_audio = create_melody()

# Duración de una repetición (10 segundos)
repeat_duration = 10 * 1000  # 10 segundos en ms

# Ajuste de volumen global más bajo todavía
melody_audio = melody_audio.apply_gain(-6)  # Más silencioso general

# Calcular número de repeticiones
num_repeats = total_duration_seconds // 10

# Crear la pista final repitiendo
final_audio = AudioSegment.silent(duration=0)
for _ in range(num_repeats):
    final_audio += melody_audio



from datetime import datetime
now = datetime.now()
formatted_time = now.strftime("%Y%m%d%H%M%S")

# Exportar
final_audio.export("melody_super_tetrica"+formatted_time+".wav", format="wav")

print(f"Melodía tétrica profunda generada en 'melody_super_tetrica.wav' con {total_duration_seconds} segundos.")

Melodía tétrica profunda generada en 'melody_super_tetrica.wav' con 20 segundos.


# Duración mejorada

In [ ]:

# Función para crear una nota con frecuencia específica (en Hz) y duración (en milisegundos)
def create_note(frequency, duration):
    return Sine(frequency).to_audio_segment(duration=duration).fade_out(100)  # pequeño fade para evitar chasquido

# Notas musicales más graves
notes = {
    'C2': 65.41,
    'C#2': 69.30,
    'D2': 73.42,
    'D#2': 77.78,
    'E2': 82.41,
    'F2': 87.31,
    'F#2': 92.50,
    'G2': 98.00,
    'G#2': 103.83,
    'A2': 110.00,
    'A#2': 116.54,
    'B2': 123.47,
    'C3': 130.81,
    'D3': 146.83, #Mas agudos
}

# Duraciones
note_duration = 1200  # en milisegundos
pause_duration = 50  # silencio entre notas
total_duration_seconds = 60  # duración total deseada
volume_db = 0  # volumen más bajo por nota

# # Melodía tétrica con tonos graves
# melody = [
#     'C2', 'D2', 'D#2', 'D2', 'C2', 'C2', 'C2', 'D2',
#     'D#2', 'E2', 'F2', 'C2', 'D2', 'E2', 'D2', 'C2',
#     'A2', 'A2', 'G#2', 'G2', 'F2', 'D2', 'C2'
# ]

# # Lamento Subterráneo
# melody = [
#     'A2', 'G2', 'F2', 'E2', 'F2', 'G2',
#     'A2', 'C3', 'D3', 'C3', 'A2',
#     'F2', 'E2', 'D2', 'C2', 'C2',
#     'D2', 'F2', 'E2', 'D2'
# ]

# Ritual Olvidado
# melody = [
#     'C2', 'D#2', 'C2', 'G#2', 'C2',
#     'F2', 'D#2', 'F2', 'G2', 'C3',
#     'C2', 'D#2', 'C2', 'G2', 'F2',
#     'D#2', 'C2', 'C2'
# ]

#Eco del Martillo Ritual Gusta
# melody = [
#     'C2', 'C2', 'D2', 'C2', 'C2', 'D2',
#     'E2', 'D2', 'C2', 'C2', 'F2', 'C2',
#     'E2', 'C2', 'D2', 'C2'
# ]

# Pulso de la Cripta
# melody = [
#     'F2', 'C3', 'F2', 'C3', 'E2', 'D3',
#     'F2', 'C3', 'F2', 'C3', 'D2', 'C3',
#     'F2', 'E2', 'C2', 'C2'
# ]

# Pulso de la Cripta Mejorada
melody = [
    'F2', 'C#2', 'F2', 'C#2', 'E2', 'A2',
    'F2', 'C#2', 'F2', 'C#2', 'D2', 'C#2',
    'F2', 'E2', 'C2', 'C2'
]



# Función para crear una sola instancia de la melodía
def create_melody():
    melody_audio = AudioSegment.silent(duration=0)
    #pause = AudioSegment.silent(duration=pause_duration)
    for note in melody:
        freq = notes[note]
        note_audio = create_note(freq, note_duration).apply_gain(volume_db)
        melody_audio += note_audio #+ pause
    return melody_audio

# Crear la melodía base
melody_audio = create_melody()
melody_duration_ms = len(melody_audio)
total_duration_ms = total_duration_seconds * 1000

# Calcular cuántas repeticiones caben en la duración total
num_repeats = total_duration_ms // melody_duration_ms

# Crear la pista final con repeticiones exactas
final_audio = melody_audio * int(num_repeats)

# Agregar silencio para rellenar si falta un poco
remaining = total_duration_ms - len(final_audio)
if remaining > 0:
    final_audio += AudioSegment.silent(duration=remaining)



from datetime import datetime
now = datetime.now()
formatted_time = now.strftime("%Y%m%d%H%M%S")

# Exportar
final_audio.export("melody_tetrica_final"+formatted_time+".wav", format="wav")
print(f"Melodía tétrica generada con duración exacta de {total_duration_seconds} segundos.")

Melodía tétrica generada con duración exacta de 60 segundos.
